In [1]:
from datasets import load_dataset

smol_dataset = load_dataset(
    "osunlp/SMolInstruct", 
    use_selfies=True,
    insert_core_tags=False,  # loada data w/o core tags such as <SELFIES>, </SELFIES>
    # cache_dir=os.path.join(self.root, 'cache')
    )
_task = 'forward_synthesis'
smol_tr = smol_dataset["train"].filter(lambda x: x["task"] == _task)

/home/chanhui-lee/miniconda3/envs/molca/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
mol_instruction_dataset = load_dataset(
    "zjunlp/Mol-Instructions", "Molecule-oriented Instructions"
)
molinst = mol_instruction_dataset['forward_reaction_prediction']
molinst_te = molinst.filter(lambda x: "test" in x["metadata"])['input']

In [3]:

smol_tr_mol = smol_tr['raw_input'][:]
molinst_te_mol = molinst_te[:]

In [4]:
smol_tr_mol[0], molinst_te_mol[0]

('[C][C][C][O][C][Ring1][Branch1].[C][C][N][Branch1][Ring1][C][C][C][C].[C][S][=Branch1][C][=O][=Branch1][C][=O][Cl].[C][S][Branch1][C][C][=O].[N][C@@H1][C][C][=C][C][=C][Branch2][Ring1][#Branch1][C][N][C][=C][Branch1][Ring1][C][O][C][Branch1][=Branch2][C][Branch1][C][F][Branch1][C][F][F][=N][Ring1][O][C][=C][Ring2][Ring1][C][C][Ring2][Ring1][Branch1]',
 '[C][C][I].[O][=C][Branch1][C][O][C][=C][C][=C][C][=C][Ring1][=Branch1][Br].[C][N][Branch1][C][C][C][=O].[O].[O][=C][Branch1][C][O-1][O].[Na+1]')

In [5]:
from tqdm import tqdm
def check_duplication(smol_tr_mol, molinst_te_mol, idxs, dup_idx, proc_id):
    iter_bar = tqdm(range(len(idxs)))
    checked_dup = []
    for i in tqdm(iter_bar):
        if smol_tr_mol[idxs[i]] in molinst_te_mol:
            checked_dup.append(idxs[i])
    dup_idx.extend(checked_dup)

In [11]:
import multiprocessing as mp
import numpy as np

num_procs = 250
dup_idx = mp.Manager().list()
procs = []
data = smol_tr_mol
indicies = np.arange(len(data))
chuncked_idx = np.array_split(indicies, num_procs)
for i in range(num_procs):
    proc = mp.Process(target=check_duplication, args=(data, molinst_te_mol, chuncked_idx[i], dup_idx, i))
    procs.append(proc)
    proc.start()
for proc in procs:
    proc.join()

100%|██████████| 3888/3888 [00:38<00:00, 102.04it/s] 


In [12]:
len(list(dup_idx))

128

In [14]:
data[516]

'[C][C][C][C][N][C][Branch1][C][C][=C][Branch1][C][C][S][/C][Ring1][#Branch1][=C][\\C][=Branch1][C][=O][C][=C][C][Branch1][C][Cl][=C][C][=C][Ring1][#Branch1][N].[C][C][O][C][=Branch1][C][=O][Cl].[C][C][O][C][Branch1][C][C][=O].[O].[O][=C][Branch1][C][O-1][O].[Na+1]'

In [15]:
a= list(dup_idx)
# sort
a.sort()
a

[516,
 1625,
 5204,
 5484,
 8453,
 12022,
 14108,
 20091,
 28556,
 41075,
 48148,
 55971,
 72391,
 75073,
 76547,
 102229,
 104617,
 108421,
 110488,
 126920,
 127765,
 129350,
 135515,
 152923,
 155452,
 159996,
 175525,
 177160,
 177238,
 177556,
 187344,
 194923,
 196156,
 202930,
 204779,
 209471,
 212739,
 225259,
 227995,
 230873,
 232836,
 237552,
 239683,
 246930,
 252772,
 260125,
 279452,
 290255,
 297084,
 299571,
 334665,
 335065,
 341629,
 343866,
 351574,
 359737,
 363490,
 376523,
 378651,
 381502,
 389460,
 389536,
 398719,
 399400,
 428415,
 457569,
 479704,
 479938,
 486478,
 511892,
 515764,
 522018,
 530600,
 530976,
 533657,
 537915,
 549170,
 568789,
 570043,
 581524,
 600141,
 614336,
 620084,
 621313,
 625235,
 629242,
 629748,
 636313,
 646954,
 653926,
 657333,
 664258,
 679136,
 701108,
 701495,
 704075,
 707382,
 709285,
 709919,
 733542,
 738240,
 746956,
 747749,
 748724,
 753329,
 765680,
 766535,
 781904,
 781909,
 785353,
 791437,
 795809,
 803026,
 825